# Install Libraries

In [7]:
!pip install fastapi nest-asyncio uvicorn transformers omegaconf torch

# Create Config File

In [8]:
yaml_config = """
translate_model: "vinai/vinai-translate-en2vi"
cache_dir: "./models_cache"
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [9]:
import json

# Tạo CSDL RAG mẫu (knowledge_base.json) cho dự án Smart Tourism
knowledge = {
    "The seafood is fresh": "Hải sản ở đây được đánh bắt trong ngày, rất tươi ngon.",
    "I want to find a traditional museum in Da Nang": "Bảo tàng Điêu khắc Chăm là một lựa chọn tuyệt vời tại Đà Nẵng.",
    "The tour guide was very helpful and friendly": "Hướng dẫn viên của chúng tôi luôn sẵn sàng hỗ trợ và rất thân thiện."
}
with open("knowledge_base.json", "w", encoding="utf-8") as f:
    json.dump(knowledge, f, ensure_ascii=False)

# Build Model

In [10]:
import torch
from omegaconf import OmegaConf
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class VinAITranslator:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        print("Đang nạp mô hình dịch thuật VinAI...")

        self.tokenizer = AutoTokenizer.from_pretrained(self.config.translate_model, src_lang="en_XX", cache_dir=self.config.cache_dir)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.config.translate_model, cache_dir=self.config.cache_dir)

    def __call__(self, en_text):
        inputs = self.tokenizer(en_text, return_tensors="pt", padding=True, truncation=True, max_length=1024)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                decoder_start_token_id=self.tokenizer.lang_code_to_id["vi_VN"],
                num_return_sequences=1,
                num_beams=5,
                early_stopping=True,
                max_length=1024
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# Initialize Model

In [11]:
translator = VinAITranslator("./config.yaml")

# Test thử
print(translator("The tour guide was very helpful and friendly."))

Đang nạp mô hình dịch thuật VinAI...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Hướng dẫn viên rất hữu ích và thân thiện.


# Initialize API

In [14]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading
import uvicorn

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class TranslationRequest(BaseModel):
    message: str

@app.get('/')
async def root():
    return {"message": "API Dịch Thuật Anh-Việt (VinAI) - Sinh viên: Quang"}

@app.get('/health')
async def health():
    return {"status": "ok", "model": "vinai-translate-en2vi"}

@app.post('/predict')
async def predict(data: TranslationRequest):
    if not data.message or len(data.message.strip()) == 0:
        raise HTTPException(status_code=400, detail="Vui lòng nhập câu tiếng Anh cần dịch!")

    try:
        translated_text = translator(data.message)
        return {
            "original_text": data.message,
            "translated_text": translated_text
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server Translation đã chạy tại cổng 8000")

Server Translation đã chạy tại cổng 8000


INFO:     Started server process [6796]
INFO:     Waiting for application startup.


# Call Local API

In [15]:
import requests

API_URL = "http://127.0.0.1:8000/predict"
payload = {"message": "I want to find a traditional museum in Da Nang"}

response = requests.post(API_URL, json=payload)
print("Mã trạng thái:", response.status_code)
print("Kết quả JSON:", response.json())

INFO:     127.0.0.1:51942 - "POST /predict HTTP/1.1" 200 OK
Mã trạng thái: 200
Kết quả JSON: {'original_text': 'I want to find a traditional museum in Da Nang', 'translated_text': 'Tôi muốn tìm một bảo tàng truyền thống ở Đà Nẵng'}


# Call Public API

In [16]:
import requests

# 1. Dán đường link Pinggy của ông vào đây và thêm đuôi /predict
PUBLIC_API_URL = "http://sgsny-34-125-202-207.run.pinggy-free.link/predict"

# 2. Dữ liệu thử nghiệm số 2 (Đề bài yêu cầu test ít nhất 2 dữ liệu đầu vào)
payload = {"message": "The tour guide was very helpful and friendly"}

# 3. Gọi API
response = requests.post(PUBLIC_API_URL, json=payload)

print("Mã trạng thái:", response.status_code)
print("Kết quả JSON:", response.json())

INFO:     34.125.202.207:0 - "POST /predict HTTP/1.1" 200 OK
Mã trạng thái: 200
Kết quả JSON: {'original_text': 'The tour guide was very helpful and friendly', 'translated_text': 'Hướng dẫn viên rất hữu ích và thân thiện'}
